In [1]:
import h5py, glob, os
import pathlib as pl
import geopandas as gpd
import shapely as shp
%matplotlib inline
import matplotlib.pyplot as plt

In [2]:
#if you need to go to the parent directory
os.chdir('..')
from src.hdf import *
os.chdir('..')

In [3]:
home = pl.Path(os.getcwd())
print('home is at ',home)

project = 'wy_fy22'

inputs = home/'4_v5_ready_for_cloud'
outputs_base = home/'_code'/'outputs'/project
outputs_perimeter = outputs_base/'perims'
outputs_rr = outputs_base/'rregions'

home is at  U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation


In [4]:
#get a list of the hdf files for processing
hdfs = glob.glob(str(inputs)+'/*/*.g*.hdf')

In [5]:
len(hdfs)

63

In [6]:
for hdf in hdfs:
    #geo hdf
    hf_geo = h5py.File(str(hdf),'r')
    #get domain names
    domain_geo = str(list(hf_geo['Geometry']['2D Flow Areas']['Attributes'])[0][0]).strip("'b\'")
    
    #get global projection for all geometric operations. Assume the same between model plans
    crs_prj = str(hf_geo.attrs['Projection']).strip("'b\'")
    #get domain
    geo = gpd.GeoSeries(geometry.Polygon(map(geometry.Point, hf_geo['Geometry']['2D Flow Areas']['Polygon Points'][:]))) 
    ds_area = gpd.GeoDataFrame({'name':domain_geo,'geometry':geo}, crs=crs_prj)
    out_name = hdf.split('\\')[-1].split('.')[0]
    ds_area.to_file(outputs_perimeter/f'{out_name}.shp')
    
    ##### for refinement regions
    if '2D Flow Area Refinement Regions' in hf_geo['Geometry'].keys():
        print(f'***{out_name} has a refinement region available')
        coords_list = hf_geo['Geometry']['2D Flow Area Refinement Regions']['Polygon Points'][:].tolist()
        vertices_count = len(coords_list)
        coords_gs = [] #list that will hold list of separate points for each polygon desired

        starter = 0
        sequence = 1

        while sequence < vertices_count:
            find_coord = coords_list[starter]
            compare_coord = coords_list[sequence]
            temp_list = []
            temp_list.append(coords_list[starter])

            while compare_coord != find_coord:
                temp_list.append(compare_coord)
                sequence += 1
                compare_coord = coords_list[sequence]

            temp_list.append(compare_coord)
            coords_gs.append(temp_list)

            starter = (sequence + 1)
            sequence += 2
        
        gdf_rr = gpd.GeoDataFrame(columns=['name','model','geometry'], geometry='geometry',crs=crs_prj)
        for i in range(0,len(coords_gs)):
            polygon = shp.geometry.Polygon(coords_gs[i])
            geo_series = gpd.GeoSeries([polygon],crs=crs_prj)
            new_gdf = gpd.GeoDataFrame({'name':domain_geo,'model':out_name,'geometry':geo_series}, crs=crs_prj)
            gdf_rr = pd.concat([gdf_rr, new_gdf], ignore_index=True) 
            
        # gdf_rr.plot()

        out_name_r = hdf.split('\\')[-1].split('.')[0]+'_rr'
        gdf_rr.to_file(outputs_rr/f'{out_name_r}.shp')
    else:
        print(f'{out_name} does not have a refinement region available')

Alkali Creek does not have a refinement region available
N Piney Ck-Grn R does not have a refinement region available
Fontenelle Creek does not have a refinement region available
Muddy Creek-Gree does not have a refinement region available
Boulder Creek does not have a refinement region available
East Fork River does not have a refinement region available
Buckhorn Canyon does not have a refinement region available
Eighteenmile Can does not have a refinement region available
Shute Creek does not have a refinement region available
Alkali Creek does not have a refinement region available
Upper Big Sandy does not have a refinement region available
Pacific Creek does not have a refinement region available
Sublettes Flat does not have a refinement region available
Lower Big Sandy does not have a refinement region available
Patrick Draw-BC does not have a refinement region available
BlackButteCreek does not have a refinement region available
LowerSaltWellsCr does not have a refinement region 

In [7]:
print('complete')

complete


## Hide

### Tentative Workflow
~1. make a geodataframe (or list?) to be the shapefile (with each row as a feature) = no multipart features~  
~2. loop through the coordinates~

~3. create a geoseries (?) first idea was a list but it will need to hold a polygon type object, see steps 5-7~
~4. with the for loop, go through the lists of coordinates until you find one that's identical to the first polygon (if coord in list then add it, which means you probably need to make a list so it can check)~
~5. create a polygon from the list of coordinates using geopandas (e.g. Polygon([coord1],[coord2]))~ 
~6. append the polygon to the geoseries...~ 
~7. then append that geoseries to the geodataframe (or maybe another geoseries if you don't want other attributes with it, not sure if you need a geodataframe to export to shp or not)~
~8. go back through the loop to iterate through the next set of coordinates~

coords_list = hf_geo['Geometry']['2D Flow Area Refinement Regions']['Polygon Points'][:].tolist()

vertices_count = len(coords_list)
coords_gs = [] # or geodataframe? or a dict?!!!!

starter = 0
sequence = 1

while sequence < vertices_count:
    find_coord = coords_list[starter]
    compare_coord = coords_list[sequence]
    temp_list = []
    temp_list.append(coords_list[starter])
    
    while compare_coord != find_coord:
        temp_list.append(compare_coord)
        sequence += 1
        compare_coord = coords_list[sequence]
    
    temp_list.append(compare_coord)
    coords_gs.append(temp_list)
    
    starter = (sequence+1)
    sequence += 2

gdf_rr = gpd.GeoDataFrame(columns=['name','model','geometry'], geometry='geometry',crs=crs_prj)
for i in range(0,len(coords_gs)):
    polygon = shp.geometry.Polygon(coords_gs[i])
    geo_series = gpd.GeoSeries([polygon],crs=crs_prj)
    new_gdf = gpd.GeoDataFrame({'name':domain_geo,'model':out_name,'geometry':geo_series}, crs=crs_prj)
    
    # new_gdf = gpd.GeoDataFrame(geometry=geo_series)
    
    gdf_rr = pd.concat([gdf_rr, new_gdf], ignore_index=True)  

gdf_rr.plot()

gdf_rr